# Módulo 2 - Segundo Punto: Previsión de Demanda (Dataset M5)

**Contexto.** Una gran cadena minorista necesita optimizar su gestión de inventario para miles de productos en múltiples tiendas y estados. Predecir bien la demanda evita a la vez el **exceso de stock** (capital inmovilizado) y las **roturas de stock / out-of-stock** (ventas perdidas).

**Objetivo.**
1. Construir un modelo de ML para la previsión de la demanda a nivel **producto-tienda-día** (horizonte de 28 días, `F1`…`F28`), manejando la alta variabilidad del nivel base y la estructura jerárquica de los datos.
2. Usar **Conformal Prediction** para cuantificar la incertidumbre del modelo y saber en qué casos es menos confiable.

**Datos (M5 / Walmart).**

| Archivo | Contenido |
|---|---|
| `sales_train_validation.csv` | 30.490 series: ventas diarias por producto-tienda (`d_1`…`d_1913`) + jerarquía (`item_id`, `dept_id`, `cat_id`, `store_id`, `state_id`). |
| `calendar.csv` | Mapea cada `d_i` a fecha real, día de la semana, mes, año, eventos y flags SNAP por estado. |
| `sell_prices.csv` | Precio de venta por producto, tienda y semana (`wm_yr_wk`). |
| `sample_submission.csv` | Formato de entrega: 28 días futuros (`F1`…`F28`) por `id`. |

## 1. Importación de librerías

Todas las dependencias del pipeline en una sola celda: manejo de datos, visualización, modelado (LightGBM) y utilidades de evaluación. `MAPIE` se usará más adelante para Conformal Prediction; si no está instalado, se puede añadir con `pip install mapie`.

In [ ]:
# ---- Manejo de datos y utilidades ----
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Visualización ----
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ---- Modelado (se usan a partir de la sección de modelado) ----
# LightGBM y MAPIE pueden no estar instalados aún en este entorno.
# El import es tolerante para que la fase de carga de datos no se bloquee.
try:
    import lightgbm as lgb
    _HAS_LGB = True
except ModuleNotFoundError:
    _HAS_LGB = False
    print("[aviso] lightgbm no instalado. Instalar con: pip install lightgbm")

try:
    from sklearn.metrics import mean_absolute_error, mean_squared_error
except ModuleNotFoundError:
    print("[aviso] scikit-learn no instalado. Instalar con: pip install scikit-learn")

# ---- Conformal Prediction (se usará en la sección de incertidumbre) ----
# from mapie.regression import MapieRegressor  # pip install mapie

# ---- Configuración general ----
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías importadas correctamente")
print("pandas:", pd.__version__, "| numpy:", np.__version__,
      "| lightgbm:", lgb.__version__ if _HAS_LGB else "no instalado")

## 2. Carga ordenada de los datos

El archivo `sales_train_validation.csv` tiene 30.490 filas y ~1.919 columnas, y `sell_prices.csv` supera los 6,8 millones de filas. Para cargar de forma eficiente definimos **tipos reducidos** (`int16`, `float32`, `category`): esto baja mucho el consumo de memoria y acelera las operaciones posteriores.

Definimos primero la ruta base y una función auxiliar para reportar el uso de memoria.

In [ ]:
# Ruta a la carpeta con los CSV (ajusta si ejecutas desde otra ubicación)
DATA_DIR = Path("Modulo_dos_segundo_punto")
if not DATA_DIR.exists():
    DATA_DIR = Path(".")  # fallback: mismos CSV en el directorio actual

assert (DATA_DIR / "sales_train_validation.csv").exists(), \
    f"No se encuentran los CSV en {DATA_DIR.resolve()}"


def mem_mb(df: pd.DataFrame) -> float:
    """Memoria usada por un DataFrame en MB."""
    return df.memory_usage(deep=True).sum() / 1024 ** 2


print("Directorio de datos:", DATA_DIR.resolve())

### 2.1 Calendario

Mapea cada día (`d_1`, `d_2`, …) a su fecha real y aporta variables de calendario (día de la semana, mes, año), eventos (festivos, deportivos, religiosos) y los flags **SNAP** (días de ayudas alimentarias por estado, que elevan la demanda de FOODS).

In [ ]:
calendar_dtypes = {
    "wm_yr_wk": "int32",
    "wday": "int8",
    "month": "int8",
    "year": "int16",
    "snap_CA": "int8",
    "snap_TX": "int8",
    "snap_WI": "int8",
    "weekday": "category",
    "event_name_1": "category",
    "event_type_1": "category",
    "event_name_2": "category",
    "event_type_2": "category",
}

calendar = pd.read_csv(
    DATA_DIR / "calendar.csv",
    dtype=calendar_dtypes,
    parse_dates=["date"],
)

print(f"calendar: {calendar.shape[0]:,} filas x {calendar.shape[1]} columnas | {mem_mb(calendar):.2f} MB")
print(f"Rango de fechas: {calendar['date'].min().date()} -> {calendar['date'].max().date()}")
calendar.head()

### 2.2 Precios de venta

Precio semanal de cada producto en cada tienda. Es el archivo más grande (~6,8M filas), por eso se cargan `store_id` e `item_id` como `category` y el precio como `float32`.

In [ ]:
prices_dtypes = {
    "store_id": "category",
    "item_id": "category",
    "wm_yr_wk": "int32",
    "sell_price": "float32",
}

sell_prices = pd.read_csv(DATA_DIR / "sell_prices.csv", dtype=prices_dtypes)

print(f"sell_prices: {sell_prices.shape[0]:,} filas x {sell_prices.shape[1]} columnas | {mem_mb(sell_prices):.2f} MB")
sell_prices.head()

### 2.3 Ventas diarias (nivel base)

Las 30.490 series producto-tienda, con sus 5 columnas de identificación (jerarquía) y las ventas diarias `d_1`…`d_1913`. Convertimos las columnas de identificación a `category` y todas las columnas `d_*` a `int16` (las ventas diarias son valores pequeños), lo que reduce drásticamente la memoria.

In [ ]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

# Leemos solo el header para construir el mapa de tipos de todas las columnas d_*
header = pd.read_csv(DATA_DIR / "sales_train_validation.csv", nrows=0).columns.tolist()
d_cols = [c for c in header if c.startswith("d_")]

sales_dtypes = {c: "category" for c in id_cols}
sales_dtypes.update({c: "int16" for c in d_cols})

sales = pd.read_csv(DATA_DIR / "sales_train_validation.csv", dtype=sales_dtypes)

print(f"sales: {sales.shape[0]:,} filas x {sales.shape[1]:,} columnas | {mem_mb(sales):.2f} MB")
print(f"Días de histórico: {len(d_cols)} (d_1 ... {d_cols[-1]})")
sales[id_cols + d_cols[:5]].head()

### 2.4 Formato de entrega

`sample_submission` define qué hay que predecir: para cada `id` (30.490 `_validation` + 30.490 `_evaluation`) los 28 días futuros `F1`…`F28`.

In [ ]:
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
print(f"submission: {submission.shape[0]:,} filas x {submission.shape[1]} columnas")
print("Horizonte de predicción:", [c for c in submission.columns if c.startswith('F')][:3], "...",
      [c for c in submission.columns if c.startswith('F')][-1])
submission.head()

### 2.5 Resumen de la carga

Verificación rápida de que todo quedó cargado y ordenado: dimensiones de la jerarquía y memoria total.

In [ ]:
print("=== Dimensiones de la jerarquía (nivel base) ===")
print("Estados     :", sales['state_id'].cat.categories.tolist())
print("Categorías  :", sales['cat_id'].cat.categories.tolist())
print("Departamentos:", sales['dept_id'].cat.categories.tolist())
print("Tiendas     :", sales['store_id'].cat.categories.tolist())
print("Productos únicos:", sales['item_id'].nunique())
print("Series base (producto-tienda):", sales['id'].nunique())

total_mem = mem_mb(calendar) + mem_mb(sell_prices) + mem_mb(sales) + mem_mb(submission)
print(f"\nMemoria total en RAM: {total_mem:,.1f} MB")

## 3. Análisis exploratorio (EDA)

Antes de modelar necesitamos entender la **dificultad** del problema. En demanda minorista a nivel producto-tienda el reto principal es la **intermitencia** (muchos días con cero ventas) y la **estructura jerárquica**. Esta sección responde a:

1. ¿Qué tan intermitentes son las series? (% de ceros).
2. ¿Cómo se reparte la demanda entre categorías, estados y tiendas?
3. ¿Hay estacionalidad semanal/anual y efecto de eventos y SNAP?

Las respuestas justifican las decisiones de modelado y métrica que vienen después.

### 3.1 Intermitencia de la demanda

Medimos el porcentaje de ceros global y por serie. Un porcentaje alto implica demanda intermitente, lo que condiciona tanto el modelo (debe tolerar muchos ceros) como la métrica de error.

In [ ]:
vals = sales[d_cols].to_numpy()

pct_zeros_global = (vals == 0).mean()
zeros_per_serie = (vals == 0).mean(axis=1)
mean_per_serie = vals.mean(axis=1)

print(f"% global de ceros (nivel base)        : {pct_zeros_global*100:.1f}%")
print(f"Series con > 90% de días en cero      : {(zeros_per_serie > 0.9).mean()*100:.1f}%")
print(f"Series con > 50% de días en cero      : {(zeros_per_serie > 0.5).mean()*100:.1f}%")
print("\nVenta diaria media por serie (unidades):")
print(pd.Series(mean_per_serie).describe().round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(zeros_per_serie, bins=40, color="#4C72B0", edgecolor="white")
ax[0].set_title("Distribución del % de ceros por serie")
ax[0].set_xlabel("Proporción de días en cero")
ax[0].set_ylabel("Nº de series")
ax[1].hist(np.clip(mean_per_serie, 0, 10), bins=40, color="#55A868", edgecolor="white")
ax[1].set_title("Venta diaria media por serie (recortada a 10)")
ax[1].set_xlabel("Unidades/día promedio")
ax[1].set_ylabel("Nº de series")
plt.tight_layout()
plt.show()

### 3.2 Reparto de la demanda en la jerarquía

Ventas totales por categoría, estado y tienda. Esto muestra qué segmentos concentran el volumen y motiva el uso de la jerarquía en el modelo (features de agregación).

In [ ]:
tot_cat = sales.groupby("cat_id", observed=True)[d_cols].sum().sum(axis=1).sort_values(ascending=False)
tot_state = sales.groupby("state_id", observed=True)[d_cols].sum().sum(axis=1).sort_values(ascending=False)
tot_store = sales.groupby("store_id", observed=True)[d_cols].sum().sum(axis=1).sort_values(ascending=False)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
tot_cat.plot(kind="bar", ax=ax[0], color="#4C72B0", title="Ventas totales por categoría")
tot_state.plot(kind="bar", ax=ax[1], color="#55A868", title="Ventas totales por estado")
tot_store.plot(kind="bar", ax=ax[2], color="#C44E52", title="Ventas totales por tienda")
for a in ax:
    a.set_ylabel("Unidades acumuladas")
    a.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print("Participación por categoría (%):")
print((tot_cat / tot_cat.sum() * 100).round(1).to_string())

### 3.3 Estacionalidad y efecto de eventos / SNAP

Construimos la serie agregada total (suma de todas las tiendas y productos) para observar tendencia y estacionalidad anual, y comparamos la demanda media en días con/sin evento y con/sin SNAP. Los flags SNAP (ayudas alimentarias) elevan sobre todo la demanda de FOODS.

In [ ]:
# Serie agregada total por día, alineada con el calendario
daily_total = sales[d_cols].sum(axis=0)
daily_total.index = daily_total.index.map(
    calendar.set_index("d")["date"].to_dict()
)
daily_total = daily_total.sort_index()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily_total.index, daily_total.values, lw=0.7, color="#4C72B0")
ax.plot(daily_total.index, daily_total.rolling(28).mean().values, lw=2, color="#C44E52",
        label="Media móvil 28 días")
ax.set_title("Ventas totales diarias (todas las tiendas y productos)")
ax.set_ylabel("Unidades/día")
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
plt.show()

# Estacionalidad semanal (día de la semana) usando wday del calendario
dow = calendar.set_index("d").loc[d_cols, "wday"].to_numpy()
dow_mean = pd.Series(daily_total.to_numpy(), index=dow).groupby(level=0).mean()
wday_labels = {1: "Sáb", 2: "Dom", 3: "Lun", 4: "Mar", 5: "Mié", 6: "Jue", 7: "Vie"}
print("Venta total media por día de la semana:")
print(dow_mean.rename(index=wday_labels).round(0).to_string())

## 4. Reshape a formato largo (melt)

El modelo de ML necesita un **panel largo**: una fila por `(serie, día)` en lugar de una fila por serie con 1.913 columnas. Pasamos de formato ancho a largo con `melt` y enriquecemos cada fila con el calendario (fecha, día de semana, mes, eventos, SNAP) y el precio de venta de esa semana.

> **Nota de memoria.** El panel completo son 30.490 × 1.913 ≈ **58,3 millones de filas**. Para que el notebook sea reproducible en un portátil, el melt se hace sobre una **muestra configurable de series** (`N_SERIES`). El pipeline es idéntico para el dataset completo: basta subir `N_SERIES` a 30.490 si se dispone de memoria suficiente (o procesar por tienda en bloques).

### 4.1 Melt de ventas y unión con calendario y precios

In [ ]:
# Nº de series a incluir en el panel largo. 30490 = dataset completo.
N_SERIES = 3000

if N_SERIES >= len(sales):
    sales_sub = sales
else:
    # Muestra estratificada por tienda para conservar la jerarquía
    sales_sub = (
        sales.groupby("store_id", observed=True, group_keys=False)
        .apply(lambda g: g.sample(max(1, int(round(N_SERIES * len(g) / len(sales)))),
                                  random_state=RANDOM_STATE), include_groups=True)
    )

print(f"Series incluidas en el panel: {len(sales_sub):,} de {len(sales):,}")

# Columnas de calendario que aportan señal predictiva
cal_cols = ["d", "date", "wm_yr_wk", "wday", "month", "year",
            "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI"]

# Wide -> long
long = sales_sub.melt(id_vars=id_cols, value_vars=d_cols, var_name="d", value_name="sales")
long["sales"] = long["sales"].astype("int16")

# Unir calendario (fecha, estacionalidad, eventos, SNAP)
long = long.merge(calendar[cal_cols], on="d", how="left")

# Unir precio semanal por (store_id, item_id, wm_yr_wk)
long = long.merge(sell_prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")

print(f"Panel largo: {long.shape[0]:,} filas x {long.shape[1]} columnas | {mem_mb(long):.1f} MB")
long.head()

### 4.2 Filtrado por disponibilidad del producto

Un precio nulo tras el merge significa que el producto **todavía no estaba a la venta** en esa tienda esa semana (en M5 los productos se dan de alta en fechas distintas). Esos ceros no son demanda real sino ausencia del producto en el surtido, así que se descartan las filas anteriores a la primera semana con precio de cada serie. Así evitamos que el modelo aprenda ceros artificiales.

In [ ]:
antes = len(long)
pct_na = long["sell_price"].isna().mean() * 100
print(f"Filas con precio nulo (producto no listado aún): {pct_na:.1f}%")

# Descartar el periodo previo a la primera venta/precio de cada serie
long = long.sort_values(["id", "date"])
long = long[long["sell_price"].notna()].copy()

print(f"Filas tras filtrar: {len(long):,} (se eliminaron {antes - len(long):,}, "
      f"{(antes - len(long))/antes*100:.1f}%)")
print(f"% de ceros restante (demanda real intermitente): {(long['sales']==0).mean()*100:.1f}%")
long.head()

### 4.3 Panel listo para feature engineering

Resumen del panel largo resultante. En la siguiente fase se construyen las variables predictivas (lags, medias móviles, codificación de eventos y jerarquía) sobre esta tabla.

In [ ]:
print("Rango temporal del panel:", long["date"].min().date(), "->", long["date"].max().date())
print("Series en el panel      :", long["id"].nunique())
print("Columnas                :", long.columns.tolist())
print(f"\nMemoria del panel largo : {mem_mb(long):.1f} MB")
long.describe(include="number").round(3)

## 5. Feature engineering

Transformamos el panel largo en una matriz de variables predictivas. Las features capturan los tres patrones que vimos en el EDA: **autocorrelación** (lags y medias móviles), **estacionalidad** (calendario) y **drivers de demanda** (eventos, SNAP, precio).

**Regla clave contra la fuga de información (data leakage).** El objetivo es predecir 28 días hacia adelante. Por eso toda feature derivada de las ventas pasadas se calcula con un desfase mínimo de 28 días (`shift(28)`): en el momento de predecir el día *t+28* solo podemos usar información disponible hasta el día *t*. Las medias y desviaciones móviles se construyen **sobre `shift(28)`**, nunca sobre el valor actual.

### 5.1 Variables de rezago (lags) y estadísticos móviles

- **Lags 7, 14, 28:** ventas de hace 1, 2 y 4 semanas (capturan recurrencia semanal).
- **Medias y desviaciones móviles (ventanas 7 y 28) sobre `shift(28)`:** nivel y volatilidad reciente de la demanda sin mirar el futuro.

In [ ]:
long = long.sort_values(["id", "date"]).reset_index(drop=True)
g_sales = long.groupby("id", observed=True)["sales"]

# Lags directos
for lag in [7, 14, 28]:
    long[f"lag_{lag}"] = g_sales.shift(lag).astype("float32")

# Estadísticos móviles calculados sobre shift(28) para no filtrar futuro
for w in [7, 28]:
    base = g_sales.shift(28)
    long[f"rmean_{w}"] = base.rolling(w).mean().astype("float32")
    long[f"rstd_{w}"] = base.rolling(w).std().astype("float32")

print("Lags y móviles creados.")
print("% de NaN (inicio de cada serie, se filtran al entrenar):")
print(long[["lag_7", "lag_14", "lag_28", "rmean_7", "rmean_28"]].isna().mean().round(3).to_string())

### 5.2 Variables de calendario y drivers de demanda

- **Calendario:** día de la semana, fin de semana, semana del año, mes (estacionalidad).
- **SNAP:** se selecciona el flag del estado al que pertenece cada serie (`snap_CA/TX/WI`).
- **Evento:** indicador binario de día con evento (festivo, deportivo, religioso).

In [ ]:
# Calendario
long["dayofweek"] = long["date"].dt.dayofweek.astype("int8")
long["is_weekend"] = (long["dayofweek"] >= 5).astype("int8")
long["week"] = long["date"].dt.isocalendar().week.astype("int16")

# SNAP del estado correspondiente a cada serie
state = long["state_id"].astype(str)
long["snap"] = np.select(
    [state == "CA", state == "TX", state == "WI"],
    [long["snap_CA"], long["snap_TX"], long["snap_WI"]],
    default=0,
).astype("int8")

# Evento binario
long["is_event"] = long["event_name_1"].notna().astype("int8")

print("Variables de calendario y drivers creadas.")
print("Demanda media según SNAP (0=no, 1=sí):")
print(long.groupby("snap", observed=True)["sales"].mean().round(3).to_string())
print("\nDemanda media según evento:")
print(long.groupby("is_event", observed=True)["sales"].mean().round(3).to_string())

### 5.3 Variables de precio

El precio es un driver directo de la demanda. Añadimos el **cambio de precio** respecto a la semana anterior (promociones) y el **precio relativo** a la media histórica de la serie (caro/barato respecto a su propio nivel).

In [ ]:
pg = long.groupby("id", observed=True)["sell_price"]
long["price_change"] = (long["sell_price"] / pg.shift(1) - 1).astype("float32")
long["price_rel_mean"] = (long["sell_price"] / pg.transform("mean")).astype("float32")

print("Variables de precio creadas.")
long[["sell_price", "price_change", "price_rel_mean"]].describe().round(3)

### 5.4 Codificación de la jerarquía y matriz final

Las columnas jerárquicas (`item_id`, `dept_id`, `cat_id`, `store_id`, `state_id`) se dejan como `category`: LightGBM las maneja de forma nativa, lo que incorpora la estructura jerárquica sin necesidad de one-hot. Definimos la lista de features y el target, y verificamos el conjunto listo para entrenar.

In [ ]:
# Categóricas jerárquicas (nativas en LightGBM)
cat_features = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
for c in cat_features:
    long[c] = long[c].astype("category")

# Features numéricas y binarias
num_features = [
    "lag_7", "lag_14", "lag_28",
    "rmean_7", "rstd_7", "rmean_28", "rstd_28",
    "sell_price", "price_change", "price_rel_mean",
    "dayofweek", "is_weekend", "week", "month",
    "snap", "is_event",
]

FEATURES = cat_features + num_features
TARGET = "sales"

print(f"Total de features: {len(FEATURES)} ({len(cat_features)} categóricas + {len(num_features)} numéricas)")
print("Categóricas:", cat_features)
print("Numéricas  :", num_features)
print(f"\nPanel con features: {long.shape[0]:,} filas x {long.shape[1]} columnas | {mem_mb(long):.1f} MB")
long[["id", "date", TARGET] + num_features].head()

## 6. Modelado con LightGBM

Entrenamos un **gradient boosting** (LightGBM), el enfoque de referencia en la competencia M5 para este tipo de datos tabulares con demanda intermitente y jerarquía.

Decisiones clave:
- **Objetivo `tweedie`:** la distribución Tweedie modela bien variables no negativas con muchos ceros y cola larga, exactamente el perfil de la demanda intermitente que vimos en el EDA (68% de ceros). Es el objetivo que mejor funcionó en M5.
- **Validación temporal:** los **últimos 28 días** del histórico se reservan como validación (mismo horizonte que hay que predecir). Nunca se mezcla futuro en el entrenamiento.
- **Categóricas nativas:** las columnas jerárquicas se pasan como `categorical_feature`, sin one-hot.
- **Early stopping:** detiene el entrenamiento cuando la métrica de validación deja de mejorar, evitando sobreajuste.

### 6.1 Split temporal train / validación

Eliminamos las filas con lags/móviles aún no definidos (inicio de cada serie) y partimos por fecha: todo lo anterior al corte (`última fecha − 28 días`) es entrenamiento; los 28 días finales son validación.

In [ ]:
if not _HAS_LGB:
    raise RuntimeError("LightGBM no está instalado en este entorno. Ejecuta: pip install lightgbm scikit-learn")

# Descartar filas sin histórico suficiente para las features de rezago
model_df = long.dropna(subset=["lag_28", "rmean_28"]).copy()

max_date = model_df["date"].max()
cutoff = max_date - pd.Timedelta(days=28)

train_df = model_df[model_df["date"] <= cutoff]
valid_df = model_df[model_df["date"] > cutoff]

print(f"Corte temporal : <= {cutoff.date()} (train) | > {cutoff.date()} ... {max_date.date()} (val)")
print(f"Filas train    : {len(train_df):,}")
print(f"Filas val (28d): {len(valid_df):,}")

### 6.2 Entrenamiento

Entrenamos con objetivo Tweedie y early stopping sobre la métrica RMSE de validación.

In [ ]:
dtrain = lgb.Dataset(train_df[FEATURES], train_df[TARGET],
                     categorical_feature=cat_features, free_raw_data=False)
dvalid = lgb.Dataset(valid_df[FEATURES], valid_df[TARGET],
                     categorical_feature=cat_features, reference=dtrain, free_raw_data=False)

params = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.1,
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": RANDOM_STATE,
}

model = lgb.train(
    params, dtrain,
    num_boost_round=1000,
    valid_sets=[dtrain, dvalid], valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
print(f"\nMejor iteración: {model.best_iteration}")

### 6.3 Evaluación

Comparamos el modelo contra un **baseline naive** (predecir la venta de hace 7 días, `lag_7`). Como la demanda es intermitente, reportamos RMSE y MAE. Un modelo útil debe batir claramente al baseline.

In [ ]:
pred_val = np.clip(model.predict(valid_df[FEATURES]), 0, None)

rmse = np.sqrt(mean_squared_error(valid_df[TARGET], pred_val))
mae = mean_absolute_error(valid_df[TARGET], pred_val)

# Baseline naive: ventas de hace 7 días
baseline = np.clip(valid_df["lag_7"].fillna(0), 0, None)
rmse_base = np.sqrt(mean_squared_error(valid_df[TARGET], baseline))
mae_base = mean_absolute_error(valid_df[TARGET], baseline)

print("              RMSE     MAE")
print(f"LightGBM    {rmse:7.4f} {mae:7.4f}")
print(f"Baseline(7) {rmse_base:7.4f} {mae_base:7.4f}")
print(f"\nMejora RMSE sobre baseline: {(1 - rmse/rmse_base)*100:.1f}%")

### 6.4 Importancia de variables

Qué features explican mejor la demanda (ganancia acumulada en los árboles). Permite validar que el modelo usa señales con sentido de negocio.

In [ ]:
imp = pd.Series(model.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
imp.head(15).iloc[::-1].plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Importancia de variables (ganancia) - Top 15")
ax.set_xlabel("Ganancia acumulada")
plt.tight_layout()
plt.show()

print(imp.head(10).round(0).to_string())

### 6.5 Predicción vs real en validación

Visualización de la calidad del ajuste a nivel agregado: sumamos predicción y real por día en la ventana de validación. A nivel base las series son muy ruidosas (intermitentes), por lo que el agregado diario muestra mejor si el modelo captura el nivel y la estacionalidad.

In [ ]:
val_plot = valid_df[["date", TARGET]].copy()
val_plot["pred"] = pred_val
agg = val_plot.groupby("date").agg(real=(TARGET, "sum"), pred=("pred", "sum"))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(agg.index, agg["real"], marker="o", ms=3, label="Real", color="#4C72B0")
ax.plot(agg.index, agg["pred"], marker="o", ms=3, label="Predicción", color="#C44E52")
ax.set_title("Demanda agregada diaria: real vs predicción (ventana de validación, 28 días)")
ax.set_ylabel("Unidades/día (suma de series)")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Conformal Prediction: cuantificación de la incertidumbre

El enunciado pide usar el framework de **Conformal Prediction** para saber **cuándo el modelo tiene mayor incertidumbre**. La idea es convertir las predicciones puntuales en **intervalos de predicción** con una **garantía de cobertura** (p. ej. el 90% de los valores reales caen dentro del intervalo), sin asumir ninguna distribución de los errores.

Usamos **Split Conformal Prediction**, el método estándar y exacto para este caso:
1. Partimos el histórico en **tres bloques temporales**: entrenamiento, **calibración** y test.
2. Entrenamos el modelo solo con el bloque de entrenamiento.
3. En el bloque de calibración medimos los **scores de no-conformidad** (error absoluto del modelo).
4. El cuantil `(1-α)` de esos scores define el **ancho del intervalo**, con cobertura garantizada.

> Nota sobre la librería. `MAPIE` implementa estos métodos, pero su API 1.x está diseñada para estimadores sklearn con su propio flujo de `fit`/`conformalize`. Como aquí entrenamos un `Booster` de LightGBM con una partición temporal específica (para no filtrar futuro), implementamos el split conformal de forma explícita: son pocas líneas, es exacto y deja ver con transparencia de dónde sale cada intervalo. El fundamento es idéntico al de `MapieRegressor` con `method="base"`.

### 7.1 Partición temporal en tres bloques

- **Entrenamiento:** hasta `max_date − 56 días`.
- **Calibración:** los 28 días intermedios (`−56` a `−28`). Nunca se usan para entrenar; sirven para medir los errores con los que se calibran los intervalos.
- **Test:** los 28 días finales, donde evaluamos la cobertura real de los intervalos.

In [ ]:
cut_test = max_date - pd.Timedelta(days=28)   # inicio del bloque de test
cut_cal = max_date - pd.Timedelta(days=56)    # inicio del bloque de calibración

train_cp = model_df[model_df["date"] <= cut_cal]
calib_cp = model_df[(model_df["date"] > cut_cal) & (model_df["date"] <= cut_test)]
test_cp = model_df[model_df["date"] > cut_test]

print(f"Entrenamiento : <= {cut_cal.date()}            | {len(train_cp):,} filas")
print(f"Calibración   : {cut_cal.date()} ... {cut_test.date()} | {len(calib_cp):,} filas")
print(f"Test          : > {cut_test.date()}            | {len(test_cp):,} filas")

### 7.2 Reentrenamiento sobre el bloque de entrenamiento

Entrenamos un modelo con los mismos hiperparámetros pero solo con el bloque de entrenamiento, usando calibración como validación para el early stopping. Así el bloque de calibración queda limpio para medir los scores de no-conformidad.

In [ ]:
dtrain_cp = lgb.Dataset(train_cp[FEATURES], train_cp[TARGET],
                        categorical_feature=cat_features, free_raw_data=False)
dcalib_cp = lgb.Dataset(calib_cp[FEATURES], calib_cp[TARGET],
                        categorical_feature=cat_features, reference=dtrain_cp, free_raw_data=False)

model_cp = lgb.train(
    params, dtrain_cp,
    num_boost_round=1000,
    valid_sets=[dcalib_cp], valid_names=["calib"],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
print(f"\nMejor iteración: {model_cp.best_iteration}")

### 7.3 Intervalos conformal (ancho fijo y adaptativo)

Calculamos dos variantes para un nivel de confianza del 90% (`α = 0.1`):

- **Conformal estándar:** score = error absoluto. El cuantil da un intervalo de **ancho constante** para todas las series.
- **Conformal normalizado:** score = error absoluto dividido por la volatilidad reciente de la serie (`rstd_28`). Produce intervalos **adaptativos**: más anchos donde la demanda es más volátil (mayor incertidumbre) y más estrechos donde es estable. Es la variante informativa para el negocio.

In [ ]:
alpha = 0.10  # nivel de significancia -> 90% de cobertura objetivo

pred_calib = np.clip(model_cp.predict(calib_cp[FEATURES]), 0, None)
pred_test = np.clip(model_cp.predict(test_cp[FEATURES]), 0, None)
y_test = test_cp[TARGET].to_numpy()


def conformal_quantile(scores, alpha):
    """Cuantil conformal con la corrección de muestra finita (n+1)."""
    n = len(scores)
    k = np.ceil((n + 1) * (1 - alpha)) / n
    return np.quantile(scores, min(k, 1.0))


# --- Estándar (ancho fijo) ---
scores_abs = np.abs(calib_cp[TARGET].to_numpy() - pred_calib)
q_abs = conformal_quantile(scores_abs, alpha)
lo_std = np.clip(pred_test - q_abs, 0, None)
hi_std = pred_test + q_abs
cov_std = ((y_test >= lo_std) & (y_test <= hi_std)).mean()

# --- Normalizado (ancho adaptativo por volatilidad de la serie) ---
sigma_cal = np.maximum(calib_cp["rstd_28"].to_numpy(), 0.5)
sigma_te = np.maximum(test_cp["rstd_28"].to_numpy(), 0.5)
scores_norm = np.abs(calib_cp[TARGET].to_numpy() - pred_calib) / sigma_cal
q_norm = conformal_quantile(scores_norm, alpha)
lo_adj = np.clip(pred_test - q_norm * sigma_te, 0, None)
hi_adj = pred_test + q_norm * sigma_te
cov_adj = ((y_test >= lo_adj) & (y_test <= hi_adj)).mean()

print(f"Nivel objetivo de cobertura: {int((1-alpha)*100)}%\n")
print(f"Conformal estándar    -> q={q_abs:.3f} | cobertura={cov_std*100:.1f}% | ancho medio={np.mean(hi_std-lo_std):.2f}")
print(f"Conformal normalizado -> q={q_norm:.3f} | cobertura={cov_adj*100:.1f}% | ancho medio={np.mean(hi_adj-lo_adj):.2f}")

### 7.4 ¿Dónde tiene el modelo mayor incertidumbre?

Agregamos el ancho del intervalo normalizado por serie: las series con **bandas más anchas** son aquellas en las que el modelo es menos confiable. Como era de esperar, son productos de alta rotación y alta volatilidad (sobre todo FOODS), donde pequeños errores relativos se traducen en mayor incertidumbre absoluta.

In [ ]:
unc = test_cp[["id", "cat_id", "store_id", TARGET]].copy()
unc["pred"] = pred_test
unc["lo"] = lo_adj
unc["hi"] = hi_adj
unc["width"] = hi_adj - lo_adj
unc["abs_err"] = np.abs(unc[TARGET] - unc["pred"])

by_serie = (unc.groupby("id", observed=True)
            .agg(cat=("cat_id", "first"), store=("store_id", "first"),
                 width=("width", "mean"), abs_err=("abs_err", "mean"),
                 mean_sales=(TARGET, "mean"))
            .sort_values("width", ascending=False))

print("Top 10 series con MAYOR incertidumbre (banda más ancha):")
print(by_serie.head(10).round(2).to_string())

# Incertidumbre media por categoría
by_cat = unc.groupby("cat_id", observed=True)["width"].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
by_cat.plot(kind="bar", ax=ax, color="#C44E52")
ax.set_title("Ancho medio del intervalo (90%) por categoría = incertidumbre")
ax.set_ylabel("Ancho medio del intervalo")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### 7.5 Visualización del intervalo para una serie

Pronóstico puntual y banda de incertidumbre del 90% frente a la demanda real, para una serie de ejemplo en la ventana de test de 28 días.

In [ ]:
ejemplo = by_serie.index[3]  # una serie con incertidumbre media-alta
mask = test_cp["id"] == ejemplo
fechas = test_cp.loc[mask, "date"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(fechas, lo_adj[mask.to_numpy()], hi_adj[mask.to_numpy()],
                color="#C44E52", alpha=0.2, label="Intervalo 90%")
ax.plot(fechas, pred_test[mask.to_numpy()], color="#C44E52", lw=2, label="Predicción")
ax.plot(fechas, y_test[mask.to_numpy()], color="#4C72B0", marker="o", ms=4, label="Real")
ax.set_title(f"Pronóstico e incertidumbre (90%) - {ejemplo}")
ax.set_ylabel("Unidades/día")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Exportación de artefactos para la app Streamlit

Guardamos todo lo que la app necesita para visualizar pronósticos e incertidumbre **sin reentrenar**: el modelo, la tabla de predicciones con intervalos en la ventana de test, y el ranking de incertidumbre por serie. Los archivos se escriben en la carpeta `app_artifacts/`.

In [ ]:
ART_DIR = Path("app_artifacts")
ART_DIR.mkdir(exist_ok=True)

# 1) Modelo entrenado
model_cp.save_model(str(ART_DIR / "lgbm_model.txt"))

# 2) Predicciones + intervalos en la ventana de test (lo que grafica la app)
pred_table = test_cp[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "date", TARGET]].copy()
pred_table["pred"] = pred_test
pred_table["lo"] = lo_adj
pred_table["hi"] = hi_adj
pred_table.rename(columns={TARGET: "real"}, inplace=True)
pred_table.to_parquet(ART_DIR / "predicciones_intervalos.parquet", index=False)

# 3) Ranking de incertidumbre por serie
by_serie.reset_index().to_parquet(ART_DIR / "incertidumbre_por_serie.parquet", index=False)

# 4) Metadatos de la calibración conformal
meta = {
    "alpha": alpha,
    "cobertura_objetivo": 1 - alpha,
    "cobertura_estandar": float(cov_std),
    "cobertura_normalizado": float(cov_adj),
    "q_abs": float(q_abs),
    "q_norm": float(q_norm),
    "n_series": int(pred_table["id"].nunique()),
    "fecha_min_test": str(pred_table["date"].min().date()),
    "fecha_max_test": str(pred_table["date"].max().date()),
}
pd.Series(meta).to_json(ART_DIR / "meta.json", indent=2)

print("Artefactos guardados en", ART_DIR.resolve())
for f in sorted(ART_DIR.iterdir()):
    print("  -", f.name, f"({f.stat().st_size/1024:.1f} KB)")